# Chapter 6: Ad Click Event Aggregation
*System Design Interview Volume 2*

## TL;DR

An ad click event aggregation system ingests **1 billion daily click events** through a **Kafka-decoupled streaming pipeline**. A **MapReduce DAG** (Map/Aggregate/Reduce nodes) computes per-minute click counts and top-N most clicked ads in near real-time. **Exactly-once processing** via distributed transactions ensures billing accuracy. The system uses a **Kappa architecture** (single stream path for both real-time and historical replay), **watermarks** to handle late events, and **periodic snapshots** for fault tolerance. End-of-day **batch reconciliation** catches any remaining discrepancies.

## Requirements

| Type | Requirement | Detail |
|---|---|---|
| Functional | Aggregate click counts | Number of clicks per ad_id in last M minutes |
| Functional | Top-N ads | Top 100 most clicked ads in last M minutes |
| Functional | Filtering | Filter aggregations by ip, user_id, country |
| Non-functional | Correctness | Data used for billing -- must be accurate |
| Non-functional | Latency | End-to-end a few minutes max |
| Non-functional | Robustness | Resilient to partial failures |
| Scale | Volume | 1B clicks/day, 2M ads, 30% YoY growth |

## Estimation: Scale and Storage

In [ ]:
# Back-of-envelope for ad click event aggregation
daily_clicks = 1_000_000_000
seconds_per_day = 86_400
avg_qps = daily_clicks / seconds_per_day
peak_qps = avg_qps * 5

print(f"Average QPS: {avg_qps:,.0f}")
print(f"Peak QPS:    {peak_qps:,.0f}")

# Storage
bytes_per_event = 100  # 0.1 KB per click event
daily_storage_gb = daily_clicks * bytes_per_event / (1024**3)
monthly_storage_tb = daily_storage_gb * 30 / 1024
print(f"\nDaily raw storage:   {daily_storage_gb:.0f} GB")
print(f"Monthly raw storage: {monthly_storage_tb:.1f} TB")

# Aggregated data is much smaller
num_ads = 2_000_000
minutes_per_day = 1_440
agg_rows_per_day = num_ads * minutes_per_day
bytes_per_agg_row = 32  # ad_id + minute + count + filter_id
agg_daily_gb = agg_rows_per_day * bytes_per_agg_row / (1024**3)
print(f"\nAggregated rows/day: {agg_rows_per_day:,.0f}")
print(f"Aggregated storage/day: {agg_daily_gb:.1f} GB")

## High-Level Design

```mermaid
graph LR
    subgraph Ingestion["Data Ingestion"]
        LW[Log Watchers]
    end

    subgraph MQ1["Message Queue 1"]
        K1["Kafka -- Raw Events"]
    end

    subgraph Processing["Aggregation Service - MapReduce DAG"]
        MAP["Map Nodes -- Partition by ad_id"]
        AGG["Aggregate Nodes -- Count per minute"]
        RED["Reduce Nodes -- Top-N merge"]
    end

    subgraph MQ2["Message Queue 2"]
        K2["Kafka -- Aggregated Results"]
    end

    subgraph Storage["Storage"]
        RAW[("Raw Data DB -- Cassandra")]
        AGGDB[("Aggregation DB -- Cassandra")]
    end

    subgraph Query["Query Layer"]
        QS["Query Service -- Dashboard"]
    end

    LW --> K1
    K1 --> MAP
    K1 --> RAW
    MAP --> AGG
    AGG --> RED
    RED --> K2
    K2 --> AGGDB
    AGGDB --> QS
```

## Deep Dive

### MapReduce DAG

The aggregation service uses a directed acyclic graph (DAG) of three node types:

| Node Type | Responsibility | Example |
|---|---|---|
| **Map** | Read events, filter/transform, partition by ad_id | ad_id % 3 routes to 3 aggregate nodes |
| **Aggregate** | Count clicks per ad_id in memory each minute | Maintain in-memory counters and top-N heaps |
| **Reduce** | Merge partial results into final output | Combine top-3 from each aggregate into global top-3 |

Nodes communicate via TCP (cross-process) or shared memory (cross-thread). Intermediate data stays in memory for speed.

### Event Time vs Processing Time

```mermaid
graph LR
    subgraph Timeline["Time Comparison"]
        ET["Event Time -- when click happened"]
        PT["Processing Time -- when server sees it"]
    end
    ET -->|"Network delay can be hours"| PT
```

| Aspect | Event Time | Processing Time |
|---|---|---|
| Accuracy | High -- reflects actual user behavior | Lower -- late events distort counts |
| Reliability | Client clocks may be wrong | Server clocks are more reliable |
| Late events | Must handle with watermarks | No special handling needed |
| Recommendation | **Use for ad billing** | Use when accuracy is less critical |

**Watermark technique**: Extend each aggregation window by an adjustable period (e.g., 15 seconds) to capture slightly delayed events. Longer watermarks improve accuracy but increase latency.

### Aggregation Windows

| Window Type | Description | Use Case |
|---|---|---|
| **Tumbling** (Fixed) | Non-overlapping, equal-length intervals | Per-minute click count (use case 1) |
| **Sliding** | Overlapping windows that slide by a step | Top N ads in last M minutes (use case 2) |
| Hopping | Similar to sliding but with discrete hops | General-purpose |
| Session | Grouped by activity gaps | User session analysis |

### Exactly-Once Processing

```mermaid
graph TB
    subgraph ExactlyOnce["Distributed Transaction - Atomic"]
        A["1. Poll events from upstream Kafka"]
        B["2. Aggregate events offset 100 to 110"]
        C["3. Send result to downstream Kafka"]
        D["4. Receive ACK from downstream"]
        E["5. Save offset 110 to external storage"]
        F["6. ACK upstream with new offset"]
    end

    A --> B --> C --> D --> E --> F
```

Steps 3-6 execute as a **distributed transaction**: if any fail, the entire transaction rolls back. This prevents both **data loss** (offset saved before result sent) and **duplication** (result sent before offset saved).

### Lambda vs Kappa Architecture

| Architecture | Paths | Pros | Cons |
|---|---|---|---|
| **Lambda** | Batch + Stream in parallel | Accurate batch layer corrects stream | Two codebases to maintain |
| **Kappa** | Single stream path | One codebase, simpler | Replay depends on stream retention |

The chapter uses **Kappa**: the recalculation service reads raw data from storage and feeds it through a **dedicated** aggregation service (separate from the live one), writing results to the same aggregation database.

### Hotspot Mitigation

In [ ]:
# Demonstrate hotspot: ads follow a power-law distribution
import random

random.seed(42)

# Simulate 10,000 click events across 100 ads (Zipf-like)
num_ads = 100
num_events = 10_000
# Weight popular ads much higher
weights = [1.0 / (i + 1) for i in range(num_ads)]
total_w = sum(weights)
probs = [w / total_w for w in weights]

clicks = [0] * num_ads
for _ in range(num_events):
    r = random.random()
    cumulative = 0
    for i, p in enumerate(probs):
        cumulative += p
        if r <= cumulative:
            clicks[i] += 1
            break

top5 = sorted(enumerate(clicks), key=lambda x: -x[1])[:5]
bottom5_avg = sum(clicks[50:]) / max(len(clicks[50:]), 1)

print("Top 5 ads by clicks (hotspot illustration):")
for rank, (ad_id, count) in enumerate(top5, 1):
    print(f"  #{rank}: ad_{ad_id:03d} = {count:,} clicks ({count/num_events*100:.1f}%)")
print(f"\nAverage clicks for bottom 50 ads: {bottom5_avg:.0f}")
print(f"Hotspot ratio (top1 / bottom avg): {top5[0][1]/bottom5_avg:.1f}x")

**Solution**: A resource manager detects overloaded aggregation nodes and dynamically allocates extra nodes to split the event load. After sub-aggregation, results are merged back.

### Fault Tolerance

Aggregation nodes periodically save **snapshots** to external storage (HDFS/S3):
- Upstream Kafka offset
- In-memory aggregation state (counts, top-N heaps)

On failure, a new node recovers from the latest snapshot and replays only events after that snapshot from Kafka.

**End-of-day reconciliation**: A batch job sorts all events by event time and re-aggregates. Any discrepancies with real-time results are flagged and corrected.

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Store raw data | Full auditability, recalculation | Large storage cost (~3 TB/month) |
| Store only aggregated | Small footprint, fast queries | Cannot debug or recalculate |
| Event time | Accurate billing | Must handle late arrivals |
| Processing time | Simpler implementation | Inaccurate for billing |
| Kappa architecture | Single codebase | Replay depends on Kafka retention |
| Lambda architecture | Accurate batch correction | Two codebases to maintain |
| Tumbling window | Simple, non-overlapping | Cannot answer sliding queries |
| Cassandra for raw data | Write-optimized, time-range queries | Operational overhead vs S3+Parquet |
| Star schema filtering | Pre-computed, fast reads | Explodes bucket count with many dimensions |

## Key Takeaways

1. **Kafka decoupling** between ingestion and aggregation allows independent scaling and absorbs traffic spikes
2. **MapReduce DAG** parallelizes aggregation across ad_ids; Map partitions, Aggregate counts, Reduce merges
3. **Event time + watermarks** balance accuracy and latency for billing-critical data
4. **Exactly-once processing** requires distributed transactions (atomic offset commit + result delivery)
5. **Kappa architecture** simplifies the system by reusing the stream path for historical replay
6. **Store both raw and aggregated data**: raw for debugging/recalculation, aggregated for fast queries
7. **Reconciliation** via end-of-day batch jobs is the safety net for any real-time inaccuracies

## Related Concepts

- [[stream-processing-pipeline]] -- Kafka-decoupled event flow for real-time aggregation
- [[mapreduce-aggregation]] -- DAG of Map/Aggregate/Reduce nodes
- [[event-time-vs-processing-time]] -- choosing timestamps and handling late events
- [[aggregation-windows]] -- tumbling and sliding window strategies
- [[exactly-once-processing]] -- distributed transactions for billing accuracy
- [[lambda-kappa-architecture]] -- batch+stream vs unified stream
- [[hotspot-mitigation]] -- dynamic resource allocation for popular ads